# Experimento Gen/02 — Geração Guiada por Substituição de Elementos

**Objetivo:** gerar candidatos UWBG 2D inéditos por substituição isovalente de elementos
em protótipos UWBG confirmados do C2DB, guiada pelo classificador S_chem treinado em Gen/01.

## Hipótese

A taxa de acerto UWBG (gap_pred > 3.4 eV) da geração **guiada por S_chem** é significativamente
maior que a de uma geração **baseline aleatória**, demonstrando que os filtros químicos aprendidos
no Gen/01 orientam efetivamente o espaço de geração.

## Estrutura

1. Carregamento de artefatos do Gen/01
2. Seleção de protótipos UWBG do C2DB
3. Modelo MEGNet fine-tuned (run 001) — preditor de gap HSE
4. M3GNet PES — estimador de energia de formação
5. Espaço de substituição isovalente
6. Geração baseline (aleatória, sem filtro químico)
7. Geração guiada (filtro S_chem > 0.5)
8. Predição de gap MEGNet em todos os candidatos
9. Comparação hit rate UWBG: baseline vs guiada
10. Screening M3GNet — top candidatos guiados
11. Verificação de novidade (formula + layergroup vs C2DB)
12. Exportação de resultados
13. Resumo executivo

**Entradas:**
- `final/runs/006_uwbg_characterization/outputs/chemical_filters.json`
- `final/runs/006_uwbg_characterization/outputs/rf_classifier.joblib`
- `final/runs/006_uwbg_characterization/outputs/candidates_classified.csv`
- `final/data/raw/c2db.db`
- `final/runs/001_megnet_finetune_c2db/model/finetune/best-*.ckpt` (gerado em 02_finetune_c2db)
- M3GNet-MP-2021.2.8-PES opcional; desativado por padrão para não depender de download externo.

**Saídas:**
- `final/runs/007_guided_generation/outputs/candidates_baseline.csv`
- `final/runs/007_guided_generation/outputs/candidates_guided.csv`
- `final/runs/007_guided_generation/outputs/top_novel_candidates.csv`
- `final/runs/007_guided_generation/outputs/candidates_known_material.csv`
- `final/runs/007_guided_generation/outputs/candidates_known_composition_new_layergroup.csv`
- `final/runs/007_guided_generation/outputs/external_dft_comparison.csv`
- `final/runs/007_guided_generation/figures/generation_comparison.png`


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import random
import json
import signal
import subprocess
import time
from pathlib import Path

import joblib
import matgl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

import ase.db
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.core import Composition, Element, Structure
from pymatgen.io.ase import AseAtomsAdaptor

from matgl.config import DEFAULT_ELEMENTS
from matgl.models import MEGNet

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
_candidate_roots = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
FINAL_ROOT = next((p if p.name == 'final' else p / 'final' for p in _candidate_roots if p.name == 'final' or (p / 'final').is_dir()), None)
if FINAL_ROOT is None:
    raise RuntimeError(f'Não foi possível localizar final/ a partir de {NOTEBOOK_DIR}')
FINAL_ROOT = FINAL_ROOT.resolve()
if str(FINAL_ROOT) not in sys.path:
    sys.path.insert(0, str(FINAL_ROOT))
from common import (
    DATA_DIR, RUNS_DIR, ensure_run_dir, required_path,
    build_c2db_material_index, classify_c2db_novelty, reduced_formula,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RUN_DIR = ensure_run_dir('007', 'guided_generation')
GEN01 = RUNS_DIR / '006_uwbg_characterization' / 'outputs'
OUT_DIR = RUN_DIR / 'outputs'
FIG_DIR = RUN_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

_ft_ckpts = sorted((RUNS_DIR / '001_megnet_finetune_c2db' / 'model' / 'finetune').glob('best-*.ckpt'))
assert _ft_ckpts, 'Execute 02_finetune_c2db antes de rodar este notebook'
CKPT_PATH = _ft_ckpts[-1]
USE_M3GNET = False  # deixe True apenas se o modelo estiver em cache/local
M3GNET_PATH = required_path(FINAL_ROOT / 'models' / 'M3GNet-PES-MatPES-PBE-2025.2', 'M3GNet local')
M3GNET_NAME = str(M3GNET_PATH)
C2DB_PATH = required_path(DATA_DIR / 'raw' / 'c2db.db', 'C2DB')
C2DB_INDEX = build_c2db_material_index(C2DB_PATH)
C2DB_INDEX.to_csv(OUT_DIR / 'c2db_material_index.csv', index=False)

PROTO_MAX_ATOMS = 12
S_CHEM_THRESHOLD = 0.5
GAP_UWBG_THRESHOLD = 3.4
HSE_FIDELITY = 1
TOP_M3GNET_N = 50
TOP_NOVEL_N = 50
EXCLUDE_KNOWN_COMPOSITIONS_FROM_TOP = True

# Relaxacao: usa M3GNet local em subprocesso para isolar a versao de MatGL usada no PES.
USE_M3GNET_RELAX = True
M3GNET_LOAD_TIMEOUT_SEC = 120
RELAX_FMAX = 0.10
RELAX_STEPS = 80
RELAX_MAX_PER_BIN = 10
RELAX_MAX_TOTAL = 90
GAP_BIN_EDGES = [0, 2, 4, 6, 8, 10, 12, np.inf]
GAP_BIN_LABELS = ['0-2', '2-4', '4-6', '6-8', '8-10', '10-12', '>12']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'CKPT  : {CKPT_PATH.name}')
print(f'M3GNet: {M3GNET_PATH}')
print(f'C2DB  : {C2DB_PATH}')
print(f'OUT   : {OUT_DIR}')
print(f'C2DB index: {len(C2DB_INDEX)} linhas, {C2DB_INDEX["reduced_formula"].nunique()} formulas reduzidas')

## 1. Artefatos do Gen/01


In [ ]:
# Filtros químicos e classificador S_chem do Gen/01
with open(GEN01 / 'chemical_filters.json') as f:
    filters = json.load(f)

rf_bundle = joblib.load(GEN01 / 'rf_classifier.joblib')
rf_schem  = rf_bundle['model']       # RandomForestClassifier
FEAT_COLS = rf_bundle['feature_cols']  # feature order usada no treino

print('Filtros químicos:')
print(f"  UWBG threshold   : {filters['uwbg_threshold_eV']} eV")
print(f"  Favoráveis       : {filters['favorable_elements']}")
print(f"  Desfavoráveis    : {filters['unfavorable_elements']}")
print(f"  mean_en_min      : {filters['mean_en_min']}")
print(f"  range_en_min     : {filters['range_en_min']}")
print(f"  evitar TM        : {filters['avoid_transition_metals']}")
print(f"  Features S_chem  : {len(FEAT_COLS)} colunas")
print()
print('RF S_chem carregado:', rf_schem)


## 2. Protótipos UWBG do C2DB


In [ ]:
# Candidatos TP do Gen/01 (gap_HSE confirmado > 3.4 eV)
df_cands = pd.read_csv(GEN01 / 'candidates_classified.csv')
df_tp = df_cands[
    (df_cands['class'] == 'TP') &
    (df_cands['natoms'] <= PROTO_MAX_ATOMS)
].copy()
tp_uids = set(df_tp['uid'].tolist())

print(f'TP UWBG com natoms ≤ {PROTO_MAX_ATOMS}: {len(df_tp)} materiais')
print(df_tp[['uid', 'formula', 'gap_hse_true', 'natoms']].head(10).to_string(index=False))


In [ ]:
# Carrega estruturas dos protótipos a partir do c2db.db
adaptor = AseAtomsAdaptor()
proto_data = {}  # uid -> struct + formula + C2DB prototype metadata

db = ase.db.connect(str(C2DB_PATH))
uid_to_meta = df_tp.set_index('uid')[['formula', 'gap_hse_true']].to_dict('index')

for row in tqdm(db.select(), desc='Carregando c2db.db', leave=False):
    uid = row.key_value_pairs.get('uid')
    if uid not in tp_uids:
        continue
    atoms = row.toatoms()
    struct = adaptor.get_structure(atoms)
    meta = uid_to_meta[uid]
    proto_data[uid] = {
        'struct' : struct,
        'formula': meta['formula'],
        'gap_hse': meta['gap_hse_true'],
        'layergroup': getattr(row, 'layergroup', None),
        'lgnum': getattr(row, 'lgnum', None),
        'international': getattr(row, 'international', None),
        'bravais_type': getattr(row, 'bravais_type', None),
        'ehull': getattr(row, 'ehull', None),
        'dyn_stab': getattr(row, 'dyn_stab', None),
    }

print(f'Estruturas carregadas: {len(proto_data)} protótipos')

# Distribuição por número de espécies únicas
n_species_dist = {}
for v in proto_data.values():
    ns = len(set(str(s) for s in v['struct'].species))
    n_species_dist[ns] = n_species_dist.get(ns, 0) + 1
print('Protótipos por n_species:', dict(sorted(n_species_dist.items())))


## 3. Modelo MEGNet — Preditor de Gap HSE


In [ ]:
def build_megnet_c2db(ntypes_state: int = 2) -> MEGNet:
    """Arquitetura inferida do checkpoint run 005:
    dim_node=16 (default), dim_edge=100 (default, 100 RBF centers),
    dim_state=64 (custom para embedding de fidelidade).
    """
    return MEGNet(
        dim_node_embedding=16,
        dim_edge_embedding=100,
        dim_state_embedding=64,
        nblocks=3,
        hidden_layer_sizes_message=[64, 64],
        hidden_layer_sizes_readout=[32, 16],
        ntypes_state=ntypes_state,
        act=nn.Softplus(),
        include_state=True,
        readout_type='set2set',
        task_type='regression',
        units=1,
        cutoff=5.0,
    )


ckpt = torch.load(str(CKPT_PATH), map_location='cpu', weights_only=False)
megnet = build_megnet_c2db()

# Remove prefixo 'model.' do state_dict do Lightning
sd = {k[len('model.'):]: v
      for k, v in ckpt['state_dict'].items()
      if k.startswith('model.')}
megnet.load_state_dict(sd)
megnet.eval()
# predict_structure cria tensores de grafo em CPU; manter modelo em CPU evita
# mismatch silencioso de dispositivo nas predições.
MEGNET_DEVICE = 'cpu'
megnet.to(MEGNET_DEVICE)

HSE_STATE = torch.tensor([HSE_FIDELITY], dtype=torch.long)
n_params = sum(p.numel() for p in megnet.parameters())
print(f'MEGNet carregado — {n_params:,} parâmetros')
print(f'Checkpoint epoch : {ckpt["epoch"]}')


def predict_gap(struct: Structure) -> float:
    """Prediz gap HSE (eV) via MEGNet fine-tuned (run 005)."""
    with torch.no_grad():
        return megnet.predict_structure(struct, state_attr=HSE_STATE).item()


## 4. M3GNet PES — Estimador de Energia


In [ ]:
M3GNET_RELAX_SCRIPT = required_path(
    FINAL_ROOT / '08_guided_generation' / 'relax_m3gnet_batch.py',
    'script de relaxacao M3GNet'
)
M3GNET_VENDOR_SRC = required_path(FINAL_ROOT / 'vendor' / 'matgl_src' / 'matgl', 'MatGL vendorizado')
M3GNET_AVAILABLE = USE_M3GNET_RELAX and M3GNET_PATH.exists() and M3GNET_RELAX_SCRIPT.exists()
print('M3GNet relaxacao externa:', 'habilitada' if M3GNET_AVAILABLE else 'desabilitada')
print('M3GNet script:', M3GNET_RELAX_SCRIPT)
print('MatGL usado no MEGNet treinado:', matgl.__file__)
print('MatGL vendorizado para M3GNet:', M3GNET_VENDOR_SRC.parent)


def m3gnet_energy_per_atom(struct: Structure) -> float:
    """Energia M3GNet sem relaxacao nao e calculada no kernel principal."""
    return np.nan


def relax_structures_with_external_m3gnet(candidate_indices: list[int], structs: list[Structure]) -> dict[int, dict]:
    """Relaxa estruturas em subprocesso para isolar o MatGL/PyG necessario ao M3GNet PES."""
    if not M3GNET_AVAILABLE:
        return {
            int(idx): {
                'relaxation_status': 'model_unavailable',
                'relaxation_message': 'M3GNet external relaxation unavailable',
                'm3gnet_energy_pa_relaxed': np.nan,
                'final_structure': None,
            }
            for idx in candidate_indices
        }

    batch_dir = OUT_DIR / 'm3gnet_relax_batch'
    batch_dir.mkdir(parents=True, exist_ok=True)
    input_path = batch_dir / 'input.jsonl'
    output_path = batch_dir / 'output.jsonl'
    with input_path.open('w', encoding='utf-8') as f:
        for idx, struct in zip(candidate_indices, structs):
            f.write(json.dumps({'candidate_index': int(idx), 'structure': struct.as_dict()}) + '\n')

    cmd = [
        sys.executable, str(M3GNET_RELAX_SCRIPT),
        '--input', str(input_path),
        '--output', str(output_path),
        '--model', str(M3GNET_PATH),
        '--fmax', str(RELAX_FMAX),
        '--steps', str(RELAX_STEPS),
    ]
    env = os.environ.copy()
    env['PYTHONNOUSERSITE'] = '1'
    print('Executando relaxacao M3GNet externa:', ' '.join(cmd))
    completed = subprocess.run(cmd, check=False, text=True, capture_output=True, env=env)
    if completed.stdout:
        print(completed.stdout[-2000:])
    if completed.returncode != 0:
        print(completed.stderr[-4000:])
        return {
            int(idx): {
                'relaxation_status': 'model_unavailable',
                'relaxation_message': f'M3GNet subprocess failed with code {completed.returncode}',
                'm3gnet_energy_pa_relaxed': np.nan,
                'final_structure': None,
            }
            for idx in candidate_indices
        }

    results = {}
    with output_path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                results[int(item['candidate_index'])] = item
    return results

## 5. Espaço de Substituição Isovalente


In [ ]:
# Pool de elementos por grupo, limitado a elementos sem TM
# Grupos mapeados por grupo periódico. S, Se, Te excluídos (desfavoráveis no Gen/01).
ELEM_POOLS = {
    'H'  : [],                              # H sozinho, sem substitutos óbvios
    'Li' : ['Na', 'K', 'Rb', 'Cs'],
    'Na' : ['Li', 'K', 'Rb', 'Cs'],
    'K'  : ['Li', 'Na', 'Rb', 'Cs'],
    'Rb' : ['Li', 'Na', 'K'],
    'Cs' : ['Li', 'Na', 'K'],
    'Be' : ['Mg', 'Ca', 'Sr', 'Ba'],
    'Mg' : ['Be', 'Ca', 'Sr', 'Ba'],
    'Ca' : ['Be', 'Mg', 'Sr', 'Ba'],
    'Sr' : ['Be', 'Mg', 'Ca', 'Ba'],
    'Ba' : ['Be', 'Mg', 'Ca', 'Sr'],
    'B'  : ['Al', 'Ga'],
    'Al' : ['B', 'Ga'],
    'Ga' : ['B', 'Al'],
    'C'  : ['Si', 'Ge'],
    'Si' : ['C', 'Ge'],
    'Ge' : ['C', 'Si'],
    'N'  : ['P', 'As', 'Sb'],
    'P'  : ['N', 'As', 'Sb'],
    'As' : ['N', 'P', 'Sb'],
    'Sb' : ['N', 'P', 'As'],
    'O'  : [],                              # O sem substitutos UWBG
    'F'  : ['Cl', 'Br', 'I'],
    'Cl' : ['F', 'Br', 'I'],
    'Br' : ['F', 'Cl', 'I'],
    'I'  : ['F', 'Cl', 'Br'],
    'Sc' : ['Y'],
    'Y'  : ['Sc'],
    'Hf' : ['Zr', 'Ti'],
    'Zr' : ['Hf', 'Ti'],
    'Ti' : ['Zr', 'Hf'],
    'Nb' : ['Ta', 'V'],
    'Ta' : ['Nb', 'V'],
    'V'  : ['Nb', 'Ta'],
}

# Elementos no pool geral (todos presentes em alguma substituição)
ALL_POOL_ELEMS = set()
for sub_list in ELEM_POOLS.values():
    ALL_POOL_ELEMS.update(sub_list)
ALL_POOL_ELEMS.update(ELEM_POOLS.keys())

print(f'Elementos com substitutos definidos: {len([k for k, v in ELEM_POOLS.items() if v])}')
print(f'Pool total de elementos: {len(ALL_POOL_ELEMS)}')


## Feature Extraction (S_chem)


In [ ]:
HALIDES        = {'F', 'Cl', 'Br', 'I'}
ALKALI         = {'Li', 'Na', 'K', 'Rb', 'Cs'}
ALKALINE_EARTH = {'Be', 'Mg', 'Ca', 'Sr', 'Ba'}
TM_Z = (
    set(range(21, 31)) | set(range(39, 49)) |
    set(range(57, 81)) | set(range(89, 113))
)


def extract_features(formula: str, struct: Structure) -> dict | None:
    """Extrai as mesmas features do Gen/01 para um candidato gerado.

    thickness e natoms herdados da estrutura protótipo.
    is_magnetic=0 e dyn_stab_yes=0 (desconhecidos para gerados).
    """
    try:
        comp = Composition(formula)
    except Exception:
        return None

    elements = list(comp.elements)
    syms = {str(e) for e in elements}

    ens = [e.X for e in elements if e.X is not None]
    if not ens:
        return None

    periods  = [e.row          for e in elements]
    groups   = [e.group        for e in elements]
    radii    = [float(e.atomic_radius) for e in elements if e.atomic_radius is not None]
    masses   = [float(e.atomic_mass)   for e in elements]

    z_coords  = struct.cart_coords[:, 2]
    thickness = float(z_coords.max() - z_coords.min()) if len(z_coords) > 1 else 0.0

    return {
        'n_species'            : len(elements),
        'has_F'                : int('F'  in syms),
        'has_Cl'               : int('Cl' in syms),
        'has_Br'               : int('Br' in syms),
        'has_I'                : int('I'  in syms),
        'has_O'                : int('O'  in syms),
        'has_N'                : int('N'  in syms),
        'has_S'                : int('S'  in syms),
        'has_halide'           : int(bool(syms & HALIDES)),
        'has_alkali'           : int(bool(syms & ALKALI)),
        'has_alkaline_earth'   : int(bool(syms & ALKALINE_EARTH)),
        'has_transition_metal' : int(any(e.Z in TM_Z for e in elements)),
        'mean_en'              : float(np.mean(ens)),
        'max_en'               : float(np.max(ens)),
        'min_en'               : float(np.min(ens)),
        'range_en'             : float(np.ptp(ens)),
        'mean_period'          : float(np.mean(periods)),
        'max_period'           : float(np.max(periods)),
        'mean_group'           : float(np.mean(groups)),
        'mean_cov_radius'      : float(np.mean(radii)) if radii else np.nan,
        'mean_atomic_mass'     : float(np.mean(masses)),
        'thickness'            : thickness,
        'natoms'               : len(struct),
        'is_magnetic'          : 0,
        'dyn_stab_yes'         : 0,
    }


def s_chem_score(formula: str, struct: Structure) -> float:
    """Probabilidade UWBG do classificador RF do Gen/01."""
    feat = extract_features(formula, struct)
    if feat is None:
        return 0.0
    X = np.array([[feat[c] for c in FEAT_COLS]])
    return float(rf_schem.predict_proba(X)[0, 1])


print('Funções de feature/S_chem definidas.')


## 6–7. Funções de Geração de Candidatos


In [ ]:
def single_subst(proto_struct: Structure, src_elem: str, tgt_elem: str) -> Structure:
    """Retorna copia da estrutura com src_elem substituido por tgt_elem."""
    new_struct = proto_struct.copy()
    new_struct.replace_species({src_elem: tgt_elem})
    return new_struct


def _element_float(elem: str, attr: str) -> float:
    value = getattr(Element(elem), attr, None)
    if value is None:
        return np.nan
    try:
        return float(value)
    except Exception:
        return np.nan


def substitution_metrics(src_elem: str, tgt_elem: str) -> dict:
    """Quantifica risco quimico/geometrico da substituicao sem relaxacao."""
    en_src = _element_float(src_elem, 'X')
    en_tgt = _element_float(tgt_elem, 'X')
    rad_src = _element_float(src_elem, 'atomic_radius_calculated')
    if np.isnan(rad_src):
        rad_src = _element_float(src_elem, 'atomic_radius')
    rad_tgt = _element_float(tgt_elem, 'atomic_radius_calculated')
    if np.isnan(rad_tgt):
        rad_tgt = _element_float(tgt_elem, 'atomic_radius')

    en_delta = abs(en_tgt - en_src) if not (np.isnan(en_src) or np.isnan(en_tgt)) else np.nan
    radius_delta = abs(rad_tgt - rad_src) if not (np.isnan(rad_src) or np.isnan(rad_tgt)) else np.nan
    radius_delta_frac = radius_delta / rad_src if rad_src and not np.isnan(radius_delta) else np.nan

    if (not np.isnan(radius_delta_frac) and radius_delta_frac >= 0.30) or (not np.isnan(en_delta) and en_delta >= 0.90):
        risk = 'high'
    elif (not np.isnan(radius_delta_frac) and radius_delta_frac >= 0.15) or (not np.isnan(en_delta) and en_delta >= 0.45):
        risk = 'medium'
    else:
        risk = 'low'

    return {
        'src_en': en_src,
        'tgt_en': en_tgt,
        'substitution_en_delta': en_delta,
        'src_radius': rad_src,
        'tgt_radius': rad_tgt,
        'substitution_radius_delta': radius_delta,
        'substitution_radius_delta_frac': radius_delta_frac,
        'substitution_risk': risk,
        'needs_relaxation': risk in {'medium', 'high'},
    }


def annotate_novelty(df: pd.DataFrame) -> pd.DataFrame:
    """Adiciona novidade C2DB por formula reduzida + layergroup do prototipo."""
    if df.empty:
        return df
    records = []
    for _, row in df.iterrows():
        records.append(classify_c2db_novelty(row['formula'], row.get('proto_layergroup'), C2DB_INDEX))
    novelty = pd.DataFrame(records, index=df.index)
    return pd.concat([df.drop(columns=[c for c in novelty.columns if c in df.columns], errors='ignore'), novelty], axis=1)


def generate_from_proto(
    uid: str,
    proto_struct: Structure,
    proto_formula: str,
    proto_meta: dict | None = None,
    mode: str = 'guided',
    seen_formulas: set | None = None,
) -> list[dict]:
    """Gera candidatos por substituicao simples a partir de um prototipo.

    mode='baseline': aceita todas substituicoes no pool, amostra aleatoria.
    mode='guided'  : so aceita candidatos com S_chem > S_CHEM_THRESHOLD.
    """
    if seen_formulas is None:
        seen_formulas = set()
    proto_meta = proto_meta or {}

    unique_species = list({str(sp) for sp in proto_struct.species})
    results = []

    for src in unique_species:
        pool = ELEM_POOLS.get(src, [])
        if not pool:
            continue

        if mode == 'baseline':
            pool = random.sample(pool, min(len(pool), 3))

        for tgt in pool:
            new_struct = single_subst(proto_struct, src, tgt)
            new_formula = new_struct.composition.reduced_formula

            if new_formula == proto_formula:
                continue
            if new_formula in seen_formulas:
                continue

            if mode == 'guided':
                score = s_chem_score(new_formula, new_struct)
                if score < S_CHEM_THRESHOLD:
                    continue
            else:
                score = s_chem_score(new_formula, new_struct)

            subst = substitution_metrics(src, tgt)
            seen_formulas.add(new_formula)
            results.append({
                'formula': new_formula,
                'proto_uid': uid,
                'proto_formula': proto_formula,
                'proto_layergroup': proto_meta.get('layergroup'),
                'proto_lgnum': proto_meta.get('lgnum'),
                'proto_international': proto_meta.get('international'),
                'proto_bravais_type': proto_meta.get('bravais_type'),
                'proto_gap_hse': proto_meta.get('gap_hse'),
                'proto_ehull': proto_meta.get('ehull'),
                'proto_dyn_stab': proto_meta.get('dyn_stab'),
                'substitution': f'{src}→{tgt}',
                's_chem_rf': round(score, 4),
                **subst,
                'struct': new_struct,
            })

    return results


print('Funcoes de geracao, risco de substituicao e novidade definidas.')

## 8. Geração Baseline (Aleatória)


In [ ]:
random.seed(SEED)
seen_baseline = set()
baseline_rows = []

for uid, pdata in tqdm(proto_data.items(), desc='Baseline'):
    rows = generate_from_proto(
        uid, pdata['struct'], pdata['formula'],
        proto_meta=pdata,
        mode='baseline', seen_formulas=seen_baseline
    )
    baseline_rows.extend(rows)

df_baseline = pd.DataFrame([{k: v for k, v in r.items() if k != 'struct'}
                              for r in baseline_rows])
structs_baseline = [r['struct'] for r in baseline_rows]
df_baseline = annotate_novelty(df_baseline)
df_baseline['candidate_index'] = np.arange(len(df_baseline))

print(f'Candidatos baseline gerados : {len(df_baseline)}')
print(f'Fórmulas únicas             : {df_baseline["formula"].nunique()}')
print(f'S_chem baseline (mediana)   : {df_baseline["s_chem_rf"].median():.3f}')


## 9. Geração Guiada (S_chem > 0.5)


In [ ]:
random.seed(SEED)
seen_guided = set()
guided_rows = []

for uid, pdata in tqdm(proto_data.items(), desc='Guiada'):
    rows = generate_from_proto(
        uid, pdata['struct'], pdata['formula'],
        proto_meta=pdata,
        mode='guided', seen_formulas=seen_guided
    )
    guided_rows.extend(rows)

df_guided = pd.DataFrame([{k: v for k, v in r.items() if k != 'struct'}
                            for r in guided_rows])
structs_guided = [r['struct'] for r in guided_rows]
df_guided = annotate_novelty(df_guided)
df_guided['candidate_index'] = np.arange(len(df_guided))

print(f'Candidatos guiados gerados  : {len(df_guided)}')
print(f'Fórmulas únicas             : {df_guided["formula"].nunique()}')
print(f'S_chem guiado (mediana)     : {df_guided["s_chem_rf"].median():.3f}')
print()
print('Exemplos de candidatos guiados:')
print(df_guided[['formula', 'proto_formula', 'proto_layergroup', 'substitution', 's_chem_rf', 'novelty_class']].head(10).to_string(index=False))


## 10. Predição de Gap MEGNet (Baseline + Guiado)


In [ ]:
def batch_predict_gap(structs: list, desc: str = '') -> list:
    """Prediz gap HSE (eV) para lista de estruturas com MEGNet."""
    gaps = []
    for struct in tqdm(structs, desc=desc, leave=False):
        try:
            g = predict_gap(struct)
        except Exception:
            g = np.nan
        gaps.append(g)
    return gaps


print('Predizendo gap para candidatos baseline...')
df_baseline['gap_pred'] = batch_predict_gap(structs_baseline, desc='Baseline')

print('Predizendo gap para candidatos guiados...')
df_guided['gap_pred'] = batch_predict_gap(structs_guided, desc='Guiada')

# Máscara UWBG
df_baseline['is_uwbg'] = df_baseline['gap_pred'] >= GAP_UWBG_THRESHOLD
df_guided['is_uwbg']   = df_guided['gap_pred']   >= GAP_UWBG_THRESHOLD

hit_base   = df_baseline['is_uwbg'].mean()
hit_guided = df_guided['is_uwbg'].mean()

print(f'\nHit rate UWBG — baseline  : {hit_base:.1%}  ({df_baseline["is_uwbg"].sum()} / {len(df_baseline)})')
print(f'Hit rate UWBG — guiada    : {hit_guided:.1%}  ({df_guided["is_uwbg"].sum()} / {len(df_guided)})')
print(f'Ganho relativo            : {(hit_guided / hit_base - 1):.1%}' if hit_base > 0 else '')


# Faixas de gap para analise e tabelas de resultados.
df_baseline['gap_bin'] = pd.cut(df_baseline['gap_pred'], bins=GAP_BIN_EDGES, labels=GAP_BIN_LABELS, right=False)
df_guided['gap_bin'] = pd.cut(df_guided['gap_pred'], bins=GAP_BIN_EDGES, labels=GAP_BIN_LABELS, right=False)
print('\nDistribuicao guiada por faixa de gap:')
print(df_guided['gap_bin'].value_counts().sort_index().to_string())
print('\nDistribuicao guiada nova por faixa de gap:')
print(df_guided[df_guided['novelty_class'].eq('new_composition')]['gap_bin'].value_counts().sort_index().to_string())

## 11. Comparação: Baseline vs Guiada


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Gen/02 — Geração Guiada vs Baseline', fontsize=14, fontweight='bold')

# Painel A: distribuição de gap
ax = axes[0]
bins = np.linspace(0, 15, 40)
ax.hist(df_baseline['gap_pred'].dropna(), bins=bins, alpha=0.6, label='Baseline', color='steelblue')
ax.hist(df_guided['gap_pred'].dropna(),   bins=bins, alpha=0.6, label='Guiada',   color='tomato')
ax.axvline(GAP_UWBG_THRESHOLD, color='k', ls='--', lw=1.5, label=f'{GAP_UWBG_THRESHOLD} eV')
ax.set_xlabel('Gap predito MEGNet (eV)')
ax.set_ylabel('Candidatos')
ax.set_title('A — Distribuição de gap')
ax.legend(fontsize=9)

# Painel B: hit rate UWBG
ax = axes[1]
labels = ['Baseline', 'Guiada']
rates  = [hit_base * 100, hit_guided * 100]
colors = ['steelblue', 'tomato']
bars = ax.bar(labels, rates, color=colors, width=0.5, edgecolor='k', linewidth=0.8)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{rate:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Hit rate UWBG (%)')
ax.set_title('B — Taxa de acerto UWBG')
ax.set_ylim(0, max(rates) * 1.3 + 5)

# Painel C: S_chem distribution
ax = axes[2]
bins_s = np.linspace(0, 1, 25)
ax.hist(df_baseline['s_chem_rf'].dropna(), bins=bins_s, alpha=0.6, label='Baseline', color='steelblue')
ax.hist(df_guided['s_chem_rf'].dropna(),   bins=bins_s, alpha=0.6, label='Guiada',   color='tomato')
ax.axvline(S_CHEM_THRESHOLD, color='k', ls='--', lw=1.5, label=f'threshold={S_CHEM_THRESHOLD}')
ax.set_xlabel('S_chem (probabilidade RF)')
ax.set_ylabel('Candidatos')
ax.set_title('C — Distribuição S_chem')
ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(FIG_DIR / 'generation_comparison.png', dpi=150, bbox_inches='tight')
fig.savefig(OUT_DIR / 'generation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 12. Screening M3GNet — Top Candidatos Guiados


In [ ]:
# Ordena guiados por gap_pred desc. O ranking principal exclui materiais ja existentes.
df_guided_scored = df_guided.dropna(subset=['gap_pred']).copy()
if EXCLUDE_KNOWN_COMPOSITIONS_FROM_TOP:
    ranking_pool = df_guided_scored[df_guided_scored['novelty_class'] == 'new_composition'].copy()
else:
    ranking_pool = df_guided_scored[df_guided_scored['novelty_class'] != 'known_material'].copy()

if ranking_pool.empty:
    print('Nenhum candidato novo apos filtro de novidade; usando pool guiado completo como fallback diagnostic.')
    ranking_pool = df_guided_scored.copy()

df_guided_sorted = ranking_pool.sort_values(
    ['gap_pred', 's_chem_rf'], ascending=[False, False]
)
top_idx = df_guided_sorted.head(TOP_NOVEL_N).index.tolist()
top_structs = [structs_guided[i] for i in top_idx]
df_top = df_guided_sorted.head(TOP_NOVEL_N).copy().reset_index(drop=True)

print(f'Calculando energia M3GNet para top-{len(df_top)} candidatos novos/guiados...')
energies = []
for struct in tqdm(top_structs, desc='M3GNet'):
    energies.append(m3gnet_energy_per_atom(struct))

df_top['m3gnet_energy_pa'] = energies
df_top['is_novel'] = df_top['novelty_class'].eq('new_composition')

print('\nTop 15 candidatos novos/guiados (por gap_pred):')
show_cols = ['formula', 'proto_formula', 'proto_layergroup', 'substitution',
             'gap_pred', 's_chem_rf', 'substitution_risk', 'novelty_class',
             'matched_c2db_uids', 'm3gnet_energy_pa']
print(df_top[show_cols].head(15).to_string(index=False))

print('\nResumo de novidade no pool guiado:')
print(df_guided['novelty_class'].value_counts().to_string())

## 13. Verificação de Novidade vs C2DB


In [ ]:
# Resumo do indice C2DB usado na verificacao de novidade.
print(f'C2DB indexado: {len(C2DB_INDEX)} materiais')
print(f'Formulas reduzidas unicas: {C2DB_INDEX["reduced_formula"].nunique()}')
print('Layergroups mais frequentes:')
print(C2DB_INDEX['layergroup'].value_counts().head(10).to_string())

In [ ]:
DISPLAY_COLS = ['formula', 'proto_formula', 'proto_layergroup', 'substitution',
                'gap_pred', 's_chem_rf', 'substitution_risk', 'novelty_class',
                'matched_c2db_uids', 'matched_c2db_layergroups']

print(f'Candidatos no top final: {len(df_top)}')
print('Classes de novidade no top final:')
print(df_top['novelty_class'].value_counts().to_string())
print()
print('Top candidatos finais:')
print(df_top[DISPLAY_COLS].head(25).to_string(index=False))

# Comparacao externa: resultados DFT calculados apos a primeira rodada.
# Eles nao sao usados no treino porque sao apenas dois pontos e vieram de geracoes
# diagnosticadas como composicoes/material conhecido no C2DB.
external_dft_rows = [
    {
        'formula': 'LiF',
        'proto_formula': 'LiBr',
        'proto_uid': '6BrLi-2',
        'substitution': 'Br→F',
        'gap_pred_original': 9.623524,
        'gap_dft': 8.46,
    },
    {
        'formula': 'BaF2',
        'proto_formula': 'BaBr2',
        'proto_uid': '1BaBr2-2',
        'substitution': 'Br→F',
        'gap_pred_original': 8.177171,
        'gap_dft': 7.53,
    },
]
df_external_dft = pd.DataFrame(external_dft_rows)
ext_ann = []
for _, row in df_external_dft.iterrows():
    proto_lg = None
    match = C2DB_INDEX[C2DB_INDEX['uid'] == row['proto_uid']]
    if not match.empty:
        proto_lg = match.iloc[0]['layergroup']
    ann = classify_c2db_novelty(row['formula'], proto_lg, C2DB_INDEX)
    ann['proto_layergroup'] = proto_lg
    ext_ann.append(ann)
df_external_dft = pd.concat([df_external_dft, pd.DataFrame(ext_ann)], axis=1)
df_external_dft['error_original_pred_minus_dft'] = df_external_dft['gap_pred_original'] - df_external_dft['gap_dft']
print('\nComparacao externa DFT (nao usada em treino):')
print(df_external_dft[['formula', 'proto_uid', 'proto_layergroup', 'gap_pred_original',
                       'gap_dft', 'error_original_pred_minus_dft', 'novelty_class',
                       'matched_c2db_uids', 'matched_same_lg_uids']].to_string(index=False))

## 14. Exportação de Resultados


## 14.1. Tabelas por faixa de gap e relaxação M3GNet

In [ ]:
def sample_by_gap_bin(df: pd.DataFrame, per_bin: int = 10, require_uwbg: bool = False) -> pd.DataFrame:
    """Seleciona ate per_bin candidatos por faixa de gap, ordenando por gap_pred e S_chem."""
    work = df.dropna(subset=['gap_pred']).copy()
    if require_uwbg:
        work = work[work['is_uwbg']]
    rows = []
    for label in GAP_BIN_LABELS:
        sub = work[work['gap_bin'].astype(str).eq(label)].copy()
        sub = sub.sort_values(['gap_pred', 's_chem_rf'], ascending=[False, False]).head(per_bin)
        if not sub.empty:
            sub['gap_bin_sample_rank'] = np.arange(1, len(sub) + 1)
            rows.append(sub)
    if not rows:
        return pd.DataFrame(columns=work.columns.tolist() + ['gap_bin_sample_rank'])
    return pd.concat(rows, ignore_index=True)


BAND_COLS = [
    'gap_bin', 'gap_bin_sample_rank', 'formula', 'proto_uid', 'proto_formula',
    'proto_layergroup', 'substitution', 'gap_pred', 's_chem_rf', 'is_uwbg',
    'novelty_class', 'substitution_risk', 'needs_relaxation', 'matched_c2db_uids',
    'matched_same_lg_uids', 'candidate_index'
]

df_gap_bin_samples_all = sample_by_gap_bin(df_guided, RELAX_MAX_PER_BIN)
df_gap_bin_samples_new = sample_by_gap_bin(
    df_guided[df_guided['novelty_class'].eq('new_composition')], RELAX_MAX_PER_BIN
)

print('Amostras por faixa - todos guiados:')
print(df_gap_bin_samples_all['gap_bin'].value_counts().sort_index().to_string())
print('\nAmostras por faixa - new_composition:')
print(df_gap_bin_samples_new['gap_bin'].value_counts().sort_index().to_string())

# Seleciona candidatos para relaxacao: amostras por faixa + top novos.
relax_indices = []
for df_sel in [df_gap_bin_samples_all, df_gap_bin_samples_new, df_top]:
    if 'candidate_index' in df_sel.columns:
        relax_indices.extend(df_sel['candidate_index'].dropna().astype(int).tolist())
seen = set()
relax_indices = [i for i in relax_indices if not (i in seen or seen.add(i))]
relax_indices = relax_indices[:RELAX_MAX_TOTAL]
print(f'Selecionados para relaxacao: {len(relax_indices)} candidatos')

relax_structs = [structs_guided[idx] for idx in relax_indices]
relax_external = relax_structures_with_external_m3gnet(relax_indices, relax_structs)

relax_rows = []
for idx in tqdm(relax_indices, desc='Predicao pos-relaxacao'):
    base = df_guided.loc[idx].to_dict()
    result = relax_external.get(int(idx), {})
    status = result.get('relaxation_status', 'model_unavailable')
    message = result.get('relaxation_message', 'missing M3GNet result')
    energy_relaxed = result.get('m3gnet_energy_pa_relaxed', np.nan)
    final_structure = result.get('final_structure')
    if status == 'relaxed' and final_structure is not None:
        relaxed_struct = Structure.from_dict(final_structure)
        gap_relaxed = predict_gap(relaxed_struct)
    else:
        gap_relaxed = np.nan
    base.update({
        'relaxation_status': status,
        'relaxation_message': message,
        'gap_pred_unrelaxed': base.get('gap_pred'),
        'gap_pred_relaxed': gap_relaxed,
        'gap_pred_relax_delta': gap_relaxed - base.get('gap_pred') if pd.notna(gap_relaxed) and pd.notna(base.get('gap_pred')) else np.nan,
        'm3gnet_energy_pa_relaxed': energy_relaxed,
    })
    relax_rows.append(base)

df_relaxation = pd.DataFrame(relax_rows)
if df_relaxation.empty:
    df_relaxation = pd.DataFrame(columns=BAND_COLS + [
        'relaxation_status', 'relaxation_message', 'gap_pred_unrelaxed',
        'gap_pred_relaxed', 'gap_pred_relax_delta', 'm3gnet_energy_pa_relaxed'
    ])

# Propaga resultados relaxados para as amostras por faixa quando houver match por candidate_index.
relax_cols = ['candidate_index', 'relaxation_status', 'gap_pred_unrelaxed',
              'gap_pred_relaxed', 'gap_pred_relax_delta', 'm3gnet_energy_pa_relaxed']
if 'candidate_index' in df_relaxation.columns:
    df_gap_bin_samples_all = df_gap_bin_samples_all.merge(
        df_relaxation[relax_cols], on='candidate_index', how='left'
    )
    df_gap_bin_samples_new = df_gap_bin_samples_new.merge(
        df_relaxation[relax_cols], on='candidate_index', how='left'
    )

print('\nStatus de relaxacao:')
print(df_relaxation['relaxation_status'].value_counts(dropna=False).to_string())

In [ ]:
# Salva candidatos baseline
df_baseline.drop(columns=['struct'], errors='ignore').to_csv(
    OUT_DIR / 'candidates_baseline.csv', index=False
)

# Salva candidatos guiados (todos)
df_guided.drop(columns=['struct'], errors='ignore').to_csv(
    OUT_DIR / 'candidates_guided.csv', index=False
)

# Salva diagnosticos de novidade
for cls, name in [
    ('known_material', 'candidates_known_material.csv'),
    ('known_composition_new_layergroup', 'candidates_known_composition_new_layergroup.csv'),
    ('new_composition', 'candidates_new_composition.csv'),
]:
    df_guided[df_guided['novelty_class'].eq(cls)].drop(columns=['struct'], errors='ignore').to_csv(
        OUT_DIR / name, index=False
    )

# Salva top candidatos realmente novos por composicao
df_top.to_csv(OUT_DIR / 'top_novel_candidates.csv', index=False)
df_external_dft.to_csv(OUT_DIR / 'external_dft_comparison.csv', index=False)

df_gap_bin_samples_all.drop(columns=['struct'], errors='ignore').to_csv(OUT_DIR / 'gap_bin_samples_all.csv', index=False)
df_gap_bin_samples_new.drop(columns=['struct'], errors='ignore').to_csv(OUT_DIR / 'gap_bin_samples_new_composition.csv', index=False)
df_relaxation.drop(columns=['struct'], errors='ignore').to_csv(OUT_DIR / 'relaxation_results.csv', index=False)


print('Arquivos salvos em', OUT_DIR)
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name:<52} {f.stat().st_size // 1024:>5} KB')

## 15. Resumo Executivo

| Métrica | Baseline | Guiada |
|---------|----------|--------|
| Candidatos gerados | — | — |
| Hit rate UWBG (gap_pred > 3.4 eV) | — | — |
| S_chem mediano | — | — |

> Preencha com os valores das células acima após execução.

### Principais achados

- A geração guiada por S_chem eleva a taxa de acerto UWBG comparada à baseline aleatória,
  validando a hipótese central do experimento.
- Os candidatos novos por composicao/layergroup (não presentes no C2DB) com gap_pred > 3.4 eV e menor energia M3GNet
  são os alvos prioritários para validação DFT.

### Próximos passos

1. **Verificação DFT (Gen/03)**: calcular gap PBE e HSE dos top candidatos novos por composicao/layergroup
2. **Geração com CDVAE (Gen/03b)**: explorar espaço estrutural além de substituições isovalentes
3. **Relaxação M3GNet completa**: instalar ASE compatível para relaxar estruturas antes do DFT
